# Identity check at cfg 1

The question this notebook answers: **does a cold clip conditioned on track T
actually sound like track T, with no guidance and near-deterministic sampling?**

Guidance is set to 1.0 on purpose. `l_null + 1.0 * (l_cond - l_null) == l_cond`,
so cfg 1 is the model's own conditional with nothing amplified. If identity is
genuinely learned it shows here. On the pre-16k checkpoint it did not: cfg 1 sat
at chance rank 3.17/6 (t = -0.44), and only cfg 8 pulled it to 1.67 -- which the
ear rejected anyway.

Low temperature removes sampling noise as an excuse. At `temperature <= 0` the
sampler is pure argmax, so every seed collapses to one clip; 0.2-0.3 keeps a
little variety while staying far below the 1.0 the probes used.

Two readouts, and the second is the one that decides:

1. **Identification** -- rank the clip's own track among the real anchors by
   timbre distance. Chance is (N+1)/2. This is a coarse spectral measure and it
   has already disagreed with the ear once, so treat it as a screen, not a verdict.
2. **Listening** -- generated and real side by side, per track. This is the test.

In [ ]:
from __future__ import annotations

import numpy as np
import torch
import torchaudio
from IPython.display import Audio, HTML, display

from ab_harness.config import load_config
from ab_harness.model.audio import normalize_lufs, to_int16
from ab_harness.worker.generator import SampleRequest
from ab_harness.worker.service import GenerationService
from probe_cross_track import profile_distance, timbre_profile

# Point this at the run to judge. "auto" takes the newest AR/DPO checkpoint.
CHECKPOINT = "saved_ar_20260906_16k_scratch/ar_latest.ckpt"

CFG_STRENGTH = 1.0   # == plain conditional, nothing amplified
TEMPERATURES = [0.2, 0.5]
SECONDS = 10.0
N_TRACKS = 6
SEEDS = [1, 2]

In [ ]:
cfg = load_config("config_ab.yaml")
cfg.generator.checkpoint = CHECKPOINT

service = GenerationService(cfg)
service.load()
corpus, tracks = service.corpus(), service._tracks
gen, dec = service._generator, service._decoder
assert gen is not None and dec is not None

frames = int(SECONDS * cfg.sampler.fps)
print(f"checkpoint {CHECKPOINT}")
print(f"window     {gen.window} positions")
print(f"corpus     {len(corpus)} tracks")

In [ ]:
import random

rng = random.Random(0)
chosen = rng.sample([t for t in corpus if t.num_frames > frames + 1], N_TRACKS)
for t in chosen:
    print(f"[{t.track_idx:>3}] {t.track_name}")

In [ ]:
def decode(tokens: np.ndarray) -> np.ndarray:
    """
    Args:
      tokens (np.ndarray): (T, R) int16 code grid.

    Returns:
      np.ndarray: (N,) float32 waveform, LUFS-normalised.
    """
    audio = normalize_lufs(dec.decode(tokens), dec.sample_rate, cfg.session.target_lufs)
    return to_int16(audio).astype(np.float32) / 32768.0


# Real audio per track, at a fixed relative offset: the target to match.
anchors, anchor_pcm = {}, {}
for t in chosen:
    s = int(max(0, min(t.num_frames - frames - 1, t.num_frames * 0.35)))
    pcm = decode(tracks[t.track_idx].tokens[s : s + frames].numpy().astype(np.int16))
    anchor_pcm[t.track_idx] = pcm
    anchors[t.track_idx] = timbre_profile(pcm, dec.sample_rate)
print(f"{len(anchors)} anchors decoded")

In [ ]:
grid = [
    (t.track_idx, temp, seed)
    for temp in TEMPERATURES
    for seed in SEEDS
    for t in chosen
]
reqs = [
    SampleRequest(
        track_idx=idx,
        style=tracks[idx].style[rng.randrange(tracks[idx].style.shape[0])],
        use_track_id=True,
        use_style=True,
        frames=frames,
        prompt=None,              # cold: conditioning is the only signal
        temperature=temp,
        top_k=250,
        top_p=0.0,
        cfg_strength=CFG_STRENGTH,
        seed=seed,
    )
    for idx, temp, seed in grid
]

codes, size = [], max(1, cfg.generator.max_batch)
for i in range(0, len(reqs), size):
    codes += gen.sample_batch(reqs[i : i + size])
    print(f"  {min(i + size, len(reqs))}/{len(reqs)}", flush=True)

In [ ]:
clips = {}
rows = []
for (idx, temp, seed), code_ in zip(grid, codes):
    pcm = decode(code_.numpy().astype(np.int16))
    clips[(idx, temp, seed)] = pcm
    dists = {k: profile_distance(timbre_profile(pcm, dec.sample_rate), v)
             for k, v in anchors.items()}
    own = dists[idx]
    rank = 1 + sum(1 for k, d in dists.items() if k != idx and d < own)
    rows.append((idx, temp, seed, rank, own, np.mean([d for k, d in dists.items() if k != idx])))

n = len(chosen)
print(f"chance rank {(n + 1) / 2:.1f}/{n}, chance rank-1 {1 / n:.0%}\n")
print(f"{'track':>6} {'temp':>5} {'seed':>5} {'rank':>6} {'d(own)':>8} {'d(other)':>9}")
for idx, temp, seed, rank, own, oth in rows:
    print(f"{idx:>6} {temp:>5.2f} {seed:>5} {rank:>4}/{n} {own:8.4f} {oth:9.4f}")

print()
for temp in TEMPERATURES:
    sel = [r for r in rows if r[1] == temp]
    ranks = np.array([r[3] for r in sel], dtype=float)
    delta = np.array([r[4] - r[5] for r in sel])
    se = delta.std(ddof=1) / np.sqrt(len(delta)) if len(delta) > 1 else np.nan
    print(f"temp {temp:.2f}  n={len(sel):>3}  mean rank {ranks.mean():.2f}  "
          f"rank-1 {(ranks == 1).mean():.0%}  own-other {delta.mean():+.4f}  "
          f"t={delta.mean() / se if se else np.nan:.2f}")

## The listening test

One row per track: the real audio first, then the generations at each
temperature. The question is not "is this good music" -- it is **"do the
generated clips belong to the same track as the anchor above them?"**

If they do not, identity is still unlearned and the metric above is measuring
spectral brightness rather than identity, exactly as it did on the previous
checkpoint.

In [ ]:
sr = dec.sample_rate
for t in chosen:
    display(HTML(f"<h3>[{t.track_idx}] {t.track_name}</h3><b>real</b>"))
    display(Audio(anchor_pcm[t.track_idx], rate=sr))
    for temp in TEMPERATURES:
        for seed in SEEDS:
            display(HTML(f"<b>cfg {CFG_STRENGTH:g} &middot; temp {temp} &middot; seed {seed}</b>"))
            display(Audio(clips[(t.track_idx, temp, seed)], rate=sr))

## Optional: is it the conditioning, or would any cold clip score this way?

The control the identification number needs. Same sampling, both streams nulled,
so the model has no track information at all. If the conditioned clips do not
beat this, nothing was transferred.

In [ ]:
null_reqs = [
    SampleRequest(
        track_idx=chosen[0].track_idx,
        style=tracks[chosen[0].track_idx].style[0],
        use_track_id=False,
        use_style=False,
        frames=frames,
        prompt=None,
        temperature=temp,
        top_k=250,
        top_p=0.0,
        cfg_strength=0.0,     # nothing to guide towards with both streams off
        seed=seed,
    )
    for temp in TEMPERATURES
    for seed in SEEDS
]
null_codes = gen.sample_batch(null_reqs)

null_ranks = []
for req, code_ in zip(null_reqs, null_codes):
    prof = timbre_profile(decode(code_.numpy().astype(np.int16)), dec.sample_rate)
    dists = sorted((profile_distance(prof, v), k) for k, v in anchors.items())
    null_ranks.append(1 + [k for _, k in dists].index(req.track_idx))
print(f"unconditional: mean rank {np.mean(null_ranks):.2f}/{n} over {len(null_ranks)} clips")
print("(both streams are off, so this should sit at chance)")

In [ ]:
service.close()